
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bluerrror/germany-hydrology/blob/main/examples/basin_explorer.ipynb)



# ---️ Basin explorer — one dashboard, nine data sources



**Run all cells, then work entirely in the dashboard below the last one:**



- pick a **basin level** and **click a catchment** — it highlights, the map

  zooms in, its **river network** (HydroRIVERS) is drawn and its **gauges**

  appear as orange markers (click one to select it);

- press **⬇ Fetch** — climate, land cover, soil, hydrology and terrain load

  for that basin;

- use **Map overlay** to drape the **hillshaded DEM**, **land cover** or

  **soil grid** directly over the map, blended with the opacity slider;

- the tabs are all plotly: climate with **variable + resolution dropdowns**,

  land-cover composition in official ESA colors, soil, the gauge hydrograph;

- the **HBV tab** calibrates the package's HBV-96 with Optuna: choose the

  **training range with a slider**, trials and objective — the hydrograph

  shows observed vs simulated with **NSE/KGE boxes** for train and test.



Everything is live — nothing here is precomputed.

In [1]:

# Setup — no-op locally; on Colab installs the plugins + widget stack

import importlib.util, sys



if importlib.util.find_spec("germany_hydrology") is None:

    %pip install -q git+https://github.com/Bluerrror/germany-hydrology ipyleaflet ipywidgets plotly anywidget optuna

if importlib.util.find_spec("earthkit_data_soilgrids") is None:

    %pip install -q git+https://github.com/Bluerrror/earthkit-data-soilgrids

if "google.colab" in sys.modules:

    from google.colab import output

    output.enable_custom_widget_manager()

In [2]:

import base64

import io

import json

import sys



import earthkit.data as ekd

import numpy as np

import pandas as pd

import plotly.express as px

import plotly.graph_objects as go

import plotly.io as pio

from matplotlib import colormaps

from matplotlib.colors import LightSource

from matplotlib.pyplot import imsave



# Render plotly self-contained so figures show inside widgets on ANY frontend:

# Colab keeps its own renderer; elsewhere use notebook_connected (plotly.js

# from CDN) so JupyterLab shows charts even without the plotly lab extension.

if "google.colab" not in sys.modules:

    pio.renderers.default = "notebook_connected"



ekd.config.set("cache-policy", "user")



from germany_hydrology import GERMANY, signatures as hs

from germany_hydrology.landcover import CLASSES



# official ESA WorldCover class colors (identity colors of the product)

WC_COLORS = {10: "#006400", 20: "#ffbb22", 30: "#ffff4c", 40: "#f096ff",

             50: "#fa0000", 60: "#b4b4b4", 70: "#f0f0f0", 80: "#0064c8",

             90: "#0096a0", 95: "#00cf75", 100: "#fae6a0"}



basins = ekd.from_source("hydrosheds", product="basins", level=7, bbox=GERMANY).to_pandas()

gauges = ekd.from_source("pegelonline").to_pandas().dropna(subset=["longitude", "latitude"])

print(f"{len(basins)} basins, {len(gauges)} gauges ready")

252 basins, 729 gauges ready



## Fetch helpers — plain functions on top of `from_source`, reusable anywhere

In [3]:

RECENT_END = (pd.Timestamp.now() - pd.Timedelta(days=6)).strftime("%Y-%m-%d")





def fetch_climate(source, basin, start="2010-01-01", end=RECENT_END, hyras_year=2020):

    lon, lat = basin.geometry.centroid.x, basin.geometry.centroid.y

    if source.startswith("ERA5"):

        return ekd.from_source(

            "era5-timeseries", latitude=lat, longitude=lon, start=start, end=end,

            variables=["precipitation_sum", "temperature_2m_mean",

                       "temperature_2m_max", "temperature_2m_min",

                       "et0_fao_evapotranspiration", "snowfall_sum"],

        ).to_pandas()

    if source.startswith("DWD"):

        st = ekd.from_source("dwd-observations").to_pandas()

        st = st[st["to_date"] > "2020"]

        sid = ((st["latitude"] - lat) ** 2 + (st["longitude"] - lon) ** 2).idxmin()

        df = ekd.from_source("dwd-observations", station=sid, period="all").to_pandas()

        df.attrs["station"] = f"{st.loc[sid, 'name']} ({sid})"

        return df.select_dtypes("number")

    w, s, e, n = basin.geometry.bounds                       # HYRAS

    return ekd.from_source("dwd-grids", variable="precipitation",

                           years=hyras_year, bbox=[w, s, e, n]).to_xarray()





def fetch_landcover(source, basin):

    w, s, e, n = basin.geometry.bounds

    da = ekd.from_source("worldcover", bbox=[w, s, e, n],

                         year=int(source.split()[-1])).to_xarray()

    da = da.rio.clip([basin.geometry], crs="EPSG:4326", drop=True)

    vals, counts = np.unique(da.values, return_counts=True)

    frac = pd.Series(counts, index=vals).drop(0, errors="ignore")   # 0 = nodata

    frac = (frac / frac.sum() * 100).round(2)

    colors = [WC_COLORS.get(v, "#999999") for v in frac.index]

    frac.index = [CLASSES.get(v, str(v)) for v in frac.index]

    return da, frac, colors





def fetch_soil(source, basin):

    w, s, e, n = basin.geometry.bounds

    if source.startswith("BÜK"):

        gdf = ekd.from_source("buek1000", bbox=[w, s, e, n]).to_pandas()

        gdf = gdf.clip(basin.geometry)

        area = gdf.to_crs("EPSG:3035").area

        frac = (area.groupby(gdf["Symbol"]).sum() / area.sum() * 100)

        return gdf, frac.sort_values(ascending=False).round(2)

    prop = {"SoilGrids clay": "clay", "SoilGrids sand": "sand",

            "SoilGrids pH": "phh2o", "SoilGrids SOC": "soc"}[source]

    da = ekd.from_source("soilgrids", property=prop, depth="0-5cm",

                         stat="mean", bbox=[w, s, e, n]).to_xarray()["band_1"]

    da = (da.where(da != 0) / 10.0).rename(prop)

    return da.rio.write_crs("EPSG:4326").rio.clip([basin.geometry], drop=True), None





def fetch_hydrology(gauge_name, parameter="W"):

    df = ekd.from_source("pegelonline", station=gauge_name,

                         parameter=parameter, start="P30D").to_pandas()

    df.index = df.index.tz_convert("UTC").tz_localize(None)

    return df





def fetch_terrain(source, basin):

    w, s, e, n = basin.geometry.bounds

    da = ekd.from_source("copernicus-dem", bbox=[w, s, e, n],

                         resolution=90 if "90" in source else 30).to_xarray()

    return da.rio.clip([basin.geometry], crs="EPSG:4326", drop=True)





def fetch_rivers(basin, max_reaches=2500):

    w, s, e, n = basin.geometry.bounds

    r = ekd.from_source("hydrosheds", product="rivers", bbox=[w, s, e, n]).to_pandas()

    r = r[r.intersects(basin.geometry)].copy()

    # big basins return tens of thousands of reaches — keep the main network

    # (highest Strahler orders) so the browser map stays responsive

    if len(r) > max_reaches and "ORD_STRA" in r.columns:

        for cutoff in range(int(r["ORD_STRA"].max()), 0, -1):

            if (r["ORD_STRA"] >= cutoff).sum() >= min(max_reaches, len(r) // 2):

                r = r[r["ORD_STRA"] >= cutoff]

                break

    r["geometry"] = r.geometry.simplify(0.002)

    return r





def fetch_hbv_data(basin, gauge_name):

    """Live discharge (~30 days) from PEGELONLINE + ERA5 forcing - a demo.

    For a real 10-70 year calibration use the web app or hbv_calibration.ipynb

    (CAMELS-DE); PEGELONLINE only serves ~30 days by design."""

    q = fetch_hydrology(gauge_name, "Q")

    q_daily = q.iloc[:, 0].resample("D").mean().dropna()

    area_km2 = float(basin.get("UP_AREA") or basin["SUB_AREA"])

    q_mm = q_daily * 86.4 / area_km2                        # m³/s -> mm/day

    lon, lat = basin.geometry.centroid.x, basin.geometry.centroid.y

    met = ekd.from_source(

        "era5-timeseries", latitude=lat, longitude=lon,

        start="2018-01-01", end=RECENT_END,

        variables=["precipitation_sum", "temperature_2m_mean",

                   "et0_fao_evapotranspiration"],

    ).to_pandas()

    return met, q_mm.reindex(met.index)


## Figures and map-overlay rendering

In [4]:

PLOT_H = 330

BLUE, ORANGE, INK = "#31688e", "#d95f02", "#444444"





def climate_figure(result, variable, freq):

    if hasattr(result, "data_vars"):                        # HYRAS Dataset

        series = result[variable].mean(dim=("x", "y")).to_series()

        title = f"HYRAS basin-mean {variable} (daily)"

    else:

        series = result[variable]

        title = f"{variable}"

    if freq != "D":

        agg = "sum" if ("precip" in variable or "snow" in variable

                        or variable in ("RSK", "RS")) else "mean"

        series = getattr(series.resample(freq), agg)()

        title += f" — {'monthly' if freq == 'MS' else 'annual'} {agg}"

    fig = px.line(x=series.index, y=series.values,

                  labels={"x": "", "y": variable}, title=title)

    fig.update_traces(line=dict(color=BLUE, width=1.5))

    fig.update_layout(height=PLOT_H, margin=dict(t=45, b=10))

    return fig





def landcover_figure(frac, colors):

    fig = go.Figure(go.Bar(x=frac.values, y=frac.index, orientation="h",

                           marker_color=colors,

                           text=[f"{v:.1f}%" for v in frac.values],

                           textposition="outside"))

    fig.update_layout(title="Land cover composition (ESA WorldCover)",

                      xaxis_title="% of basin area", height=PLOT_H,

                      margin=dict(t=45, b=10), yaxis=dict(autorange="reversed"))

    return fig





def soil_figure(result, frac, source_label):

    if frac is not None:

        top = frac.head(10)

        fig = go.Figure(go.Bar(x=top.values, y=[str(i) for i in top.index],

                               orientation="h", marker_color="#8c6d31",

                               text=[f"{v:.1f}%" for v in top.values],

                               textposition="outside"))

        fig.update_layout(title="Soil units (BÜK1000, % of basin, top 10)",

                          xaxis_title="% of basin area", height=PLOT_H,

                          margin=dict(t=45, b=10), yaxis=dict(autorange="reversed"))

        return fig

    da = result.coarsen(x=max(1, result.sizes["x"] // 350),

                        y=max(1, result.sizes["y"] // 350), boundary="trim").mean()

    fig = go.Figure(go.Heatmap(z=da.values, x=da.x.values, y=da.y.values,

                               colorscale="Viridis", colorbar_title=source_label))

    fig.update_layout(title=f"{source_label} (0–5 cm)", height=PLOT_H + 60,

                      yaxis=dict(scaleanchor="x"), margin=dict(t=45, b=10))

    return fig





def hydrology_figure(df, gauge):

    fig = px.line(x=df.index, y=df.iloc[:, 0].values,

                  labels={"x": "", "y": df.columns[0]},

                  title=f"{gauge} — {df.columns[0]}, last 30 days")

    fig.update_traces(line=dict(color=BLUE, width=1.3))

    fig.update_layout(height=PLOT_H, margin=dict(t=45, b=10))

    return fig





def _hillshade_rgba(dem):

    z = dem.values.astype(float)

    ls = LightSource(azdeg=315, altdeg=45)

    rgba = ls.shade(np.where(np.isfinite(z), z, np.nanmin(z)),

                    cmap=colormaps["terrain"], blend_mode="overlay", vert_exag=8)

    rgba[..., 3] = np.where(np.isfinite(z), 1.0, 0.0)

    return rgba





def terrain_figure(dem):

    fig = px.imshow(_hillshade_rgba(dem), x=dem.x.values, y=dem.y.values,

                    origin="upper", title="Elevation, hillshaded (Copernicus DEM)")

    fig.update_layout(height=PLOT_H + 130, margin=dict(t=45, b=10),

                      xaxis_title="Longitude", yaxis_title="Latitude")

    return fig





def hbv_figure(obs, sim, train_range, metrics):

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=obs.index, y=obs.values, name="observed",

                             line=dict(color=INK, width=2)))

    fig.add_trace(go.Scatter(x=sim.index, y=sim.values, name="HBV simulated",

                             line=dict(color=BLUE, width=2)))

    t0, t1 = train_range

    fig.add_vrect(x0=t0, x1=t1, fillcolor=BLUE, opacity=0.07, line_width=0)

    fig.add_vrect(x0=t1, x1=obs.index.max(), fillcolor=ORANGE, opacity=0.07,

                  line_width=0)

    for label, (mid, mvals) in metrics.items():

        fig.add_annotation(x=mid, y=1.14, yref="paper", showarrow=False,

                           text=(f"<b>{label}</b><br>NSE {mvals['nse']:.2f} · "

                                 f"KGE {mvals['kge']:.2f}"),

                           bgcolor="white", bordercolor="#888888",

                           borderwidth=1, font=dict(size=11))

    fig.update_layout(title="HBV — observed vs simulated (mm/day)",

                      height=PLOT_H + 60, margin=dict(t=85, b=10),

                      yaxis_title="Q (mm/day)",

                      legend=dict(orientation="h", y=-0.18))

    return fig





def _hex_to_rgb(h):

    return [int(h[i:i + 2], 16) / 255 for i in (1, 3, 5)]





def overlay_image(kind):

    """Render a fetched raster as (data-url, leaflet bounds) for the map."""

    if kind == "terrain":

        dem = basin_data["terrain"]

        rgba, da = _hillshade_rgba(dem), dem

    elif kind == "landcover":

        da = basin_data["landcover"][0]

        step = max(1, da.sizes["x"] // 1500)

        da = da[::step, ::step]

        z = da.values

        rgba = np.zeros((*z.shape, 4))

        for code, hexc in WC_COLORS.items():

            rgba[z == code] = _hex_to_rgb(hexc) + [1.0]

    else:                                                   # soil grid

        da = basin_data["soil"][0]

        z = da.values.astype(float)

        vmin, vmax = np.nanmin(z), np.nanmax(z)

        rgba = colormaps["viridis"]((z - vmin) / max(vmax - vmin, 1e-9))

        rgba[..., 3] = np.isfinite(z).astype(float)

    buf = io.BytesIO()

    imsave(buf, np.ascontiguousarray(rgba), format="png")

    url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

    bounds = ((float(da.y.min()), float(da.x.min())),

              (float(da.y.max()), float(da.x.max())))

    return url, bounds


## The dashboard — run this cell and everything happens below it

In [5]:

import ipywidgets as W

from ipyleaflet import (CircleMarker, GeoJSON, ImageOverlay, LayerGroup, Map,

                        basemaps)

from shapely.geometry import Point



state = {"basin": None, "gauge": None}

basin_data = {}



# ---------- map ---------------------------------------------------------

m = Map(center=(51.2, 10.3), zoom=6, basemap=basemaps.OpenStreetMap.Mapnik,

        scroll_wheel_zoom=True,

        layout=W.Layout(height="620px", width="54%"))



display_basins = basins[["HYBAS_ID", "geometry"]].copy()

display_basins["geometry"] = display_basins.geometry.simplify(0.01)

basin_layer = GeoJSON(data=json.loads(display_basins.to_json()),

                      style={"color": BLUE, "weight": 1, "fillOpacity": 0.04},

                      hover_style={"fillOpacity": 0.25})

highlight = GeoJSON(data={"type": "FeatureCollection", "features": []},

                    style={"color": ORANGE, "weight": 3, "fillOpacity": 0.06})

river_layer = GeoJSON(data={"type": "FeatureCollection", "features": []},

                      style={"color": BLUE, "weight": 2, "opacity": 0.85})

gauge_group = LayerGroup()

raster_group = LayerGroup()

for layer in (basin_layer, raster_group, river_layer, highlight, gauge_group):

    m.add(layer)



# ---------- controls ----------------------------------------------------

level_dd = W.Dropdown(options=[4, 5, 6, 7, 8], value=7, description="Basin level")

climate_dd = W.Dropdown(options=["ERA5 (centroid)", "DWD station (nearest)",

                                 "HYRAS grid (1 km)"], description="Climate")

lc_dd = W.Dropdown(options=["WorldCover 2021", "WorldCover 2020"], description="Land cover")

soil_dd = W.Dropdown(options=["SoilGrids clay", "SoilGrids sand", "SoilGrids pH",

                              "SoilGrids SOC", "BÜK1000 (polygons)"], description="Soil")

terrain_dd = W.Dropdown(options=["DEM 90 m", "DEM 30 m"], description="Terrain")

fetch_btn = W.Button(description="⬇ Fetch basin data", button_style="primary")

overlay_dd = W.Dropdown(options=["none", "terrain", "landcover", "soil"],

                        value="none", description="Map overlay")

opacity_sl = W.FloatSlider(value=0.75, min=0.1, max=1.0, step=0.05,

                           description="Opacity", readout_format=".0%")

status = W.HTML("<i>Click a basin to begin.</i>")



# ---------- tabs --------------------------------------------------------

# Output + fig.show() renders plotly reliably in BOTH JupyterLab and Colab;

# go.FigureWidget does not render in Colab, and display(fig) can go blank.

from IPython.display import HTML as _HTML



panels = {k: W.Output() for k in ("climate", "landcover", "soil",

                                  "hydrology", "terrain", "hbv")}

for _o in panels.values():

    with _o:

        display(_HTML("<i>fetch data first</i>"))





def show_in(name, fig_or_html):

    panels[name].clear_output(wait=True)

    with panels[name]:

        if isinstance(fig_or_html, str):

            display(_HTML(fig_or_html))

        else:

            fig_or_html.show(config={"displaylogo": False})





climate_var_dd = W.Dropdown(description="Variable", layout=W.Layout(width="300px"))

climate_freq_dd = W.Dropdown(options=[("daily", "D"), ("monthly", "MS"),

                                      ("annual", "YS")], value="MS",

                             description="Resolution", layout=W.Layout(width="220px"))

gauge_dd = W.Dropdown(description="Gauge", layout=W.Layout(width="300px"))

param_dd = W.Dropdown(options=[("water level (W)", "W"), ("discharge (Q)", "Q")],

                      value="W", description="Parameter", layout=W.Layout(width="240px"))



hbv_gauge_dd = W.Dropdown(description="Q gauge", layout=W.Layout(width="330px"))

trials_sl = W.IntSlider(value=60, min=10, max=300, step=10,

                        description="Optuna trials", continuous_update=False)

objective_dd = W.Dropdown(options=["rmse", "nse", "kge", "log_nse"], value="rmse",

                          description="Objective")

train_sl = W.SelectionRangeSlider(options=[("—", 0), ("–", 1)], index=(0, 1),

                                  description="Train range", continuous_update=False,

                                  layout=W.Layout(width="600px"))

prep_btn = W.Button(description="1) Load gauge + forcing", icon="download")

calib_btn = W.Button(description="2) Calibrate HBV", button_style="warning",

                     disabled=True)

hbv_status = W.HTML("")



tabs = W.Tab(children=[

    W.VBox([W.HBox([climate_var_dd, climate_freq_dd]), panels["climate"]]),

    panels["landcover"], panels["soil"],

    W.VBox([W.HBox([gauge_dd, param_dd]), panels["hydrology"]]),

    panels["terrain"],

    W.VBox([W.HBox([hbv_gauge_dd, prep_btn]),

            W.HBox([trials_sl, objective_dd, calib_btn]),

            train_sl, hbv_status, panels["hbv"]]),

], layout=W.Layout(width="46%"))

for i, t in enumerate(["climate", "land cover", "soil", "hydrology", "terrain", "HBV"]):

    tabs.set_title(i, t)

In [6]:

# ---------- selection: basin click, gauge markers, level switch ---------



def _gauge_marker(name, row):

    mk = CircleMarker(location=(row["latitude"], row["longitude"]), radius=6,

                      color="white", weight=1, fill_color=ORANGE, fill_opacity=0.95,

                      title=name)

    def _click(**kwargs):

        state["gauge"] = name

        gauge_dd.value = name

        status.value = f"Gauge <b>{name}</b> ({row['water']}) selected."

    mk.on_click(_click)

    return mk





def _select_basin(hybas_id):

    row = basins[basins["HYBAS_ID"] == hybas_id].iloc[0]

    state["basin"] = row

    highlight.data = json.loads(

        display_basins[display_basins["HYBAS_ID"] == hybas_id].to_json())

    w, s, e, n = row.geometry.bounds

    m.center = ((s + n) / 2, (w + e) / 2)

    m.zoom = int(np.clip(np.floor(np.log2(360 / max(e - w, (n - s) * 1.6))), 6, 11))



    inside = gauges[gauges["longitude"].between(w, e)

                    & gauges["latitude"].between(s, n)]

    inside = inside[inside.apply(

        lambda r: row.geometry.contains(Point(r["longitude"], r["latitude"])), axis=1)]

    gauge_group.clear_layers()

    for name, r in inside.iterrows():

        gauge_group.add(_gauge_marker(name, r))

    gauge_dd.options = sorted(inside.index)

    if len(inside):

        gauge_dd.value = gauge_dd.options[0]                # fires render_hydrology

        state["gauge"] = gauge_dd.value

    else:

        state["gauge"] = None

    # HBV needs discharge (Q); most gauges only publish water level (W)

    q_gauges = inside[inside["timeseries"].str.contains("Q", na=False)]

    hbv_gauge_dd.options = sorted(q_gauges.index)

    calib_btn.disabled = True

    hbv_status.value = (f"{len(q_gauges)} of {len(inside)} gauges offer discharge (Q). "

                        "Pick one and press <b>Load gauge + forcing</b>."

                        if len(q_gauges) else

                        "<b style='color:#b00'>No discharge gauge in this basin</b> — "

                        "HBV needs Q; try a larger basin (lower level).")



    status.value = f"Basin <b>{hybas_id}</b>: drawing rivers …"

    n_rivers = 0

    try:

        rivers = fetch_rivers(row)

        river_layer.data = json.loads(rivers.to_json())

        n_rivers = len(rivers)

    except Exception:

        river_layer.data = {"type": "FeatureCollection", "features": []}

    status.value = (f"Basin <b>{hybas_id}</b> selected — "

                    f"{len(inside)} gauges, {n_rivers} river reaches. "

                    "Press <b>⬇ Fetch basin data</b>.")





def on_basin_click(event=None, feature=None, **kwargs):

    if feature:

        _select_basin(feature["properties"]["HYBAS_ID"])



basin_layer.on_click(on_basin_click)





def on_level_change(change):

    global basins, display_basins

    status.value = f"Loading HydroBASINS level {change['new']} …"

    basins = ekd.from_source("hydrosheds", product="basins", level=change["new"],

                             bbox=GERMANY).to_pandas()

    display_basins = basins[["HYBAS_ID", "geometry"]].copy()

    display_basins["geometry"] = display_basins.geometry.simplify(0.01)

    basin_layer.data = json.loads(display_basins.to_json())

    for lyr in (highlight, river_layer):

        lyr.data = {"type": "FeatureCollection", "features": []}

    gauge_group.clear_layers()

    state["basin"] = None

    status.value = f"Level {change['new']}: {len(basins)} basins — click one."



level_dd.observe(on_level_change, names="value")





# ---------- fetch + tab rendering ---------------------------------------



def render_climate(change=None):

    if "climate" in basin_data and climate_var_dd.value:

        show_in("climate", climate_figure(basin_data["climate"],

                                          climate_var_dd.value,

                                          climate_freq_dd.value))



climate_var_dd.observe(render_climate, names="value")

climate_freq_dd.observe(render_climate, names="value")





def render_hydrology(change=None):

    if gauge_dd.value is None:

        return

    try:

        df = fetch_hydrology(gauge_dd.value, param_dd.value)

        basin_data["hydrology"] = df

        show_in("hydrology", hydrology_figure(df, gauge_dd.value))

    except Exception as exc:

        show_in("hydrology", f"<b style='color:#b00'>failed:</b> {exc}")



gauge_dd.observe(render_hydrology, names="value")

param_dd.observe(render_hydrology, names="value")





def refresh_overlay(change=None):

    raster_group.clear_layers()

    kind = overlay_dd.value

    if kind == "none":

        return

    key = kind if kind != "soil" else "soil"

    if key not in basin_data or (kind == "soil" and basin_data["soil"][1] is not None):

        status.value = f"<i>Fetch {kind} first (BÜK polygons cannot be draped).</i>"

        return

    url, bounds = overlay_image(kind)

    ov = ImageOverlay(url=url, bounds=bounds, opacity=opacity_sl.value)

    W.jslink((opacity_sl, "value"), (ov, "opacity"))

    raster_group.add(ov)



overlay_dd.observe(refresh_overlay, names="value")





def on_fetch(_):

    if state["basin"] is None:

        status.value = "<b style='color:#b00'>Click a basin first.</b>"

        return

    b = state["basin"]

    steps = [

        ("climate", lambda: fetch_climate(climate_dd.value, b), None),

        ("landcover", lambda: fetch_landcover(lc_dd.value, b),

         lambda r: landcover_figure(r[1], r[2])),

        ("soil", lambda: fetch_soil(soil_dd.value, b),

         lambda r: soil_figure(r[0], r[1], soil_dd.value)),

        ("terrain", lambda: fetch_terrain(terrain_dd.value, b),

         lambda r: terrain_figure(r)),

    ]

    done = []

    for name, fetcher, render in steps:

        status.value = f"Fetching <b>{name}</b> … ({' ✓ '.join(done)})"

        try:

            basin_data[name] = fetcher()

            if name == "climate":

                result = basin_data["climate"]

                opts = (list(result.data_vars) if hasattr(result, "data_vars")

                        else list(result.columns))

                climate_var_dd.options = [o for o in opts if "bnds" not in o

                                          and o != "crs" and "station" not in o]

                climate_var_dd.value = climate_var_dd.options[0]

                render_climate()

            else:

                show_in(name, render(basin_data[name]))

            done.append(name)

        except Exception as exc:

            show_in(name, f"<b style='color:#b00'>{name} failed:</b> {exc}")

    render_hydrology()

    refresh_overlay()

    basin_data["basin"] = b

    status.value = ("Fetched: <b>" + " · ".join(done) + "</b>. Try a map "

                    "overlay, or calibrate HBV in the last tab.")



fetch_btn.on_click(on_fetch)





# ---------- HBV ----------------------------------------------------------



def on_prep(_):

    gauge = hbv_gauge_dd.value

    if state["basin"] is None or gauge is None:

        hbv_status.value = ("<b style='color:#b00'>Select a basin with a "

                            "discharge (Q) gauge first.</b>")

        return

    hbv_status.value = f"Loading <b>{gauge}</b> discharge + ERA5 forcing …"

    try:

        met, obs = fetch_hbv_data(state["basin"], gauge)

    except Exception as exc:

        hbv_status.value = f"<b style='color:#b00'>Failed:</b> {exc}"

        return

    if obs.dropna().empty:

        hbv_status.value = ("<b style='color:#b00'>No usable discharge values "

                            "at this gauge right now — try another Q gauge.</b>")

        return

    basin_data["hbv_forcing"], basin_data["hbv_obs"] = met, obs

    days = obs.dropna().index

    train_sl.options = [(d.strftime("%d %b"), d) for d in days]

    train_sl.index = (0, max(1, int(len(days) * 0.6)))

    calib_btn.disabled = False

    hbv_status.value = (f"{len(days)} days of discharge "

                        f"({days.min().date()} → {days.max().date()}). Set the "

                        "training range, then calibrate. <i>~30 days is what "

                        "PEGELONLINE serves — a workflow demo; see "

                        "hbv_calibration.ipynb for a 70-year calibration.</i>")



prep_btn.on_click(on_prep)





def on_calibrate(_):

    from germany_hydrology import hbv



    met, obs = basin_data["hbv_forcing"], basin_data["hbv_obs"]

    t0, t1 = train_sl.value

    objective = ({"rmse": lambda o, s: -float(np.sqrt(np.mean((o - s) ** 2)))}

                 .get(objective_dd.value, objective_dd.value))

    hbv_status.value = (f"Calibrating: {trials_sl.value} trials, "

                        f"{objective_dd.value}, train {t0.date()} → {t1.date()} …")

    try:

        result = hbv.calibrate(

            met["precipitation_sum"], met["temperature_2m_mean"],

            met["et0_fao_evapotranspiration"], obs,

            objective=objective, n_trials=trials_sl.value, warmup=365,

            calibration_period=(str(t0.date()), str(t1.date())),

            fixed={"CFR": 0.05, "CWH": 0.1},

        )

    except Exception as exc:

        hbv_status.value = f"<b style='color:#b00'>Calibration failed:</b> {exc}"

        return

    basin_data["hbv"] = result



    days = obs.dropna()

    sim = result["simulation"].reindex(days.index)

    train = days.index <= t1

    metrics = {}

    for label, mask in [("train", train), ("test", ~train)]:

        o, s = days[mask], sim[mask]

        if len(o) >= 3 and o.std() > 0:

            metrics[label] = (o.index[len(o) // 2],

                              {"nse": hs.nse(o, s), "kge": hs.kge(o, s)})

    show_in("hbv", hbv_figure(days, sim, (t0, t1), metrics))

    hbv_status.value = ("Done — zoom into the hydrograph. Missing metric "

                        "boxes mean that period was too short/constant.")



calib_btn.on_click(on_calibrate)



dashboard = W.VBox([

    W.HBox([level_dd, climate_dd, lc_dd]),

    W.HBox([soil_dd, terrain_dd, fetch_btn]),

    W.HBox([overlay_dd, opacity_sl]),

    status,

    W.HBox([m, tabs]),

])

dashboard

(interactive widget — run the notebook to use it)


Everything fetched is in `basin_data` (pandas / xarray / GeoDataFrames),

HBV results in `basin_data["hbv"]` (`best_params`, optuna `study`, full

`simulation` series) — ready for your own analysis.



Data: DWD (CC-BY 4.0) · WSV PEGELONLINE · HydroSHEDS · BGR BÜK1000 ·

ISRIC SoilGrids (CC-BY 4.0) · ESA WorldCover (CC-BY 4.0) · Copernicus DEM

© DLR/Airbus/ESA · ERA5 via Open-Meteo (CC-BY 4.0).